# Train Bi-Encoder on FULL MathFish Dataset (A100 GPU)

**Uses all 5,956 labeled training problems from MathFish**

**IMPORTANT: Set Runtime to GPU (A100)**
- Runtime → Change runtime type → Hardware accelerator: GPU → GPU type: A100

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Clone repository from GitHub
!git clone https://github.com/matrix-mayank/mathfish.git
%cd mathfish

In [ ]:
# 3. Install dependencies
!pip install -q torch transformers sentence-transformers datasets huggingface-hub tqdm

In [ ]:
# 4. Download and prepare FULL training dataset (5,956 labeled problems)
# This filters out problems without standards labels
!python scripts/sample_train.py --n 999999 --output data/processed/problems_train_full.jsonl

# Check how many we got
!wc -l data/processed/problems_train_full.jsonl
!head -1 data/processed/problems_train_full.jsonl

In [ ]:
# 5. Run training on FULL dataset with BGE-large (SOTA encoder)
# Expected: ~60-90 minutes on H100/A100 (larger 335M param model)
# Using 15,000 steps, batch_size 32 (reduced for memory)

!python scripts/train_biencoder.py \
  --problems_file data/processed/problems_train_full.jsonl \
  --from_hf \
  --max_steps 15000 \
  --batch_size 32 \
  --num_negatives 8 \
  --temperature 0.03 \
  --early_stopping_patience 0 \
  --model_name "BAAI/bge-large-en-v1.5" \
  --fp16 \
  --output_dir outputs/biencoder_bge_large_15k

In [ ]:
# 6. Verify training completed
!ls -lh outputs/biencoder_bge_large_15k/checkpoint/

In [ ]:
# 7. Download trained model
import shutil
from google.colab import files

# Zip the checkpoint
shutil.make_archive('biencoder_bge_large_15k', 'zip', 'outputs/biencoder_bge_large_15k/checkpoint')

# Download
files.download('biencoder_bge_large_15k.zip')

print("\n✅ Download complete! Extract this on your Mac and use for inference.")

## Training Summary

**What we trained:**
- Model: **BGE-large-en-v1.5** (335M params, MTEB 70.5, SOTA open-source)
- Dataset: 5,956 labeled problems from MathFish train split
- Steps: 15,000 (~161 epochs with batch_size 32)
- Time: ~60-90 minutes on H100/A100
- Cost: ~$2-3

**Expected improvement over MiniLM-L6:**
- Previous (MiniLM-L6, 10k steps): Recall@5 = 52%, Recall@20 = 69%
- **Expected (BGE-large, 15k steps): Recall@5 = 58-65%, Recall@20 = 75-80%**

**Next steps:**
1. Download biencoder_bge_large_15k.zip
2. Run inference on full dev set (1,942 problems)
3. Compare with MiniLM-L6 baseline